# Lead-Generation (Real Estate & Construction) — Preprocessing & Categorical Encoding

This notebook loads **your actual Excel dataset** (`leadgen_realestate_construction_dataset*.xlsx`), profiles every column, and applies the encoding technique that is *appropriate for each column* — it does **not** blindly apply every technique to every column.

**Techniques demonstrated (only where they make sense):** One-Hot · Ordinal · Binary / 0-1 · Target (leakage-safe, out-of-fold) · Frequency · TF-IDF · plus Binary-base encoding (comparison) and multi-hot counts (comparison).

**Golden rule followed throughout:** the dataset is split into train/test **first**; every encoder, scaler and vectorizer is **fitted on the training set only** and then merely *applied* to the test set. The target (`conversion_status`) is never touched by anything that is applied to test data.

> **Colab usage:** upload the `.xlsx` to the Colab file panel (or just run the "Load" cell and use the upload button it shows), then *Runtime → Run all*.

## 1. Import Libraries
`category_encoders` is not pre-installed in Colab, so the cell installs it automatically (only if missing).

In [1]:
import sys, subprocess, importlib.util

def ensure(pip_name, import_name=None):
    import_name = import_name or pip_name
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

ensure("openpyxl")
ensure("scikit-learn", "sklearn")
ensure("category_encoders")

import os, re, glob, warnings
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import roc_auc_score
import category_encoders as ce

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 70)

RANDOM_STATE = 42
print("pandas", pd.__version__, "| numpy", np.__version__, "| scikit-learn", sklearn.__version__,
      "| category_encoders", ce.__version__)

pandas 3.0.2 | numpy 2.4.4 | scikit-learn 1.8.0 | category_encoders 2.11.1


## 2. Load Excel Dataset
The cell searches the usual Colab locations (current folder, `/content`, Google Drive) for any file matching `leadgen_realestate_construction_dataset*.xlsx` — so names like `... (1).xlsx` or `..__1_.xlsx` work too. If nothing is found, Colab shows an upload button.

In [2]:
PATTERN = "leadgen_realestate_construction_dataset*.xlsx"
search_dirs = [os.getcwd(), "/content", "/content/drive/MyDrive", "/mnt/user-data/uploads", "."]

def find_dataset():
    for d in search_dirs:
        hits = sorted(glob.glob(os.path.join(d, PATTERN)))
        if hits:
            return hits[0]
    hits = sorted(glob.glob(os.path.join("/content", "**", PATTERN), recursive=True))
    return hits[0] if hits else None

DATA_PATH = find_dataset()
if DATA_PATH is None:
    try:
        from google.colab import files          # only exists inside Colab
        print("Dataset not found - please upload the .xlsx file now:")
        uploaded = files.upload()
        DATA_PATH = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(f"Could not find a file matching {PATTERN!r}. Place it next to the notebook.")

print("Using file:", DATA_PATH)
xls = pd.ExcelFile(DATA_PATH)
print("Sheets:", xls.sheet_names)
df = xls.parse(xls.sheet_names[0])
print("Loaded:", df.shape)

Using file: /home/claude/nb/leadgen_realestate_construction_dataset__1_.xlsx


Sheets: ['Sheet1']


Loaded: (12000, 19)


## 3. Initial Dataset Inspection
Shape, column names, data types, missing values, number of unique values and sample rows.

In [3]:
print("Shape (rows, columns):", df.shape)
print("\nColumn names:\n", list(df.columns))

# light, label-free cleaning: strip stray whitespace in text columns (does not use the target)
for c in df.columns:
    if pd.api.types.is_string_dtype(df[c]) or df[c].dtype == object:
        df[c] = df[c].astype(str).str.strip()

profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "n_unique": df.nunique(),
    "unique_%": (df.nunique() / len(df) * 100).round(2),
    "example": df.iloc[0],
})
display(profile)
print("Total missing cells :", int(df.isna().sum().sum()))
print("Duplicate full rows :", int(df.duplicated().sum()))
print("Duplicate lead_id   :", int(df["lead_id"].duplicated().sum()))
print("\nSample rows:")
display(df.sample(6, random_state=1))

Shape (rows, columns): (12000, 19)

Column names:
 ['lead_id', 'age', 'location', 'job_role', 'lead_source', 'item_checked', 'website_visits', 'pages_viewed', 'time_on_website', 'demo_requested', 'email_response', 'days_since_last_interaction', 'avg_response_time', 'previous_purchases', 'budget', 'lead_score', 'preferred_location', 'annual_cost_of_living', 'conversion_status']


,dtype,missing,n_unique,unique_%,example
lead_id,str,0,12000,100.00,LD009064
age,int64,0,47,0.39,43
location,str,0,13,0.11,Noida
job_role,str,0,15,0.12,Civil Engineer
lead_source,str,0,12,0.10,Website
item_checked,str,0,49,0.41,Infrastructure project -> Construction project management
website_visits,int64,0,36,0.30,22
pages_viewed,int64,0,120,1.00,51
time_on_website,float64,0,1641,13.68,92.1
demo_requested,int64,0,5,0.04,0


Total missing cells : 0
Duplicate full rows : 0
Duplicate lead_id   : 0

Sample rows:


,lead_id,age,location,job_role,lead_source,item_checked,website_visits,pages_viewed,time_on_website,demo_requested,email_response,days_since_last_interaction,avg_response_time,previous_purchases,budget,lead_score,preferred_location,annual_cost_of_living,conversion_status
7163,LD002582,22,Ahmedabad,Entrepreneur,Property Portal,Infrastructure project,12,32,78.5,2,Moderate,20,9.3,1,475498000,61.3,Prahlad Nagar,530000,1
10385,LD004093,51,Ahmedabad,Business Owner,LinkedIn,Under-construction apartment,19,41,34.7,1,No Response,0,94.3,0,13094000,50.9,Satellite,450000,1
1902,LD008008,45,Ahmedabad,Civil Engineer,Referral,Industrial property,9,23,48.6,0,Moderate,17,6.3,1,90219000,43.2,Satellite,460000,0
3397,LD004938,29,Gurgaon,Civil Engineer,Referral,Luxury apartment -> Villa,7,15,22.6,0,Moderate,2,5.0,0,38359000,16.4,DLF Phase 3,845000,0
11695,LD009159,41,Hyderabad,Company Director,Facebook Ads,1 BHK apartment,17,42,77.4,0,Moderate,0,7.2,0,5809000,47.7,Nallagandla,525000,1
6443,LD007346,51,Pune,Doctor,Facebook Ads,Residential plot -> Property for investment,18,60,83.8,0,Fast,6,0.6,0,32048000,43.7,Kothrud,595000,0


## 4. Identify Feature Types
Before choosing any encoder, we look at the real values of every non-numeric column (and of numeric columns that *might* be categorical in disguise).

In [4]:
ID_COL, TARGET = "lead_id", "conversion_status"

print("lead_id examples:", df[ID_COL].head(5).tolist(), "| unique:", df[ID_COL].nunique(), "of", len(df))
for c in ["location", "job_role", "lead_source", "email_response"]:
    print(f"\n{c}  ({df[c].nunique()} unique)\n", df[c].value_counts().to_dict())

print("\ndemo_requested values:", df["demo_requested"].value_counts().sort_index().to_dict())
print("previous_purchases values:", df["previous_purchases"].value_counts().sort_index().to_dict())

pl = df["preferred_location"]
print(f"\npreferred_location: {pl.nunique()} unique; rows/category min={pl.value_counts().min()}, "
      f"median={int(pl.value_counts().median())}, max={pl.value_counts().max()}")
multi_city = df.groupby("preferred_location")["location"].nunique()
print("preferred_location values that occur in more than one city:",
      {k: sorted(df.loc[df.preferred_location == k, 'location'].unique()) for k in multi_city[multi_city > 1].index})

items_per_row = df["item_checked"].str.split(r"\s*->\s*")
all_items = sorted({i for lst in items_per_row for i in lst})
print(f"\nitem_checked: {df['item_checked'].nunique()} distinct full strings, but only {len(all_items)} distinct items "
      f"chained with '->' ({items_per_row.str.len().min()} to {items_per_row.str.len().max()} items per row)")
print("Items per row distribution:", items_per_row.str.len().value_counts().sort_index().to_dict())
print("Examples:", df["item_checked"].sample(4, random_state=3).tolist())

lead_id examples: ['LD009064', 'LD007452', 'LD008371', 'LD001858', 'LD000248'] | unique: 12000 of 12000

location  (13 unique)
 {'Bengaluru': 1631, 'Mumbai': 1508, 'Hyderabad': 1475, 'Delhi': 1304, 'Chennai': 1174, 'Pune': 1069, 'Gurgaon': 825, 'Noida': 820, 'Ahmedabad': 703, 'Kochi': 560, 'Kolkata': 356, 'Vijayawada': 339, 'Visakhapatnam': 236}

job_role  (15 unique)
 {'Software Engineer': 1838, 'Business Owner': 1134, 'Manager': 1125, 'Entrepreneur': 918, 'Government Employee': 857, 'Contractor': 743, 'Architect': 718, 'Consultant': 701, 'Doctor': 659, 'Civil Engineer': 644, 'Investor': 582, 'Sales Manager': 581, 'Real Estate Developer': 568, 'Project Manager': 482, 'Company Director': 450}

lead_source  (12 unique)
 {'Property Portal': 1851, 'Google Ads': 1684, 'Referral': 1241, 'Facebook Ads': 1236, 'Website': 1153, 'Instagram': 965, 'Direct Inquiry': 957, 'Broker': 838, 'LinkedIn': 766, 'Existing Customer': 599, 'Exhibition': 362, 'Email Campaign': 348}

email_response  (4 unique)


item_checked: 49 distinct full strings, but only 28 distinct items chained with '->' (1 to 3 items per row)
Items per row distribution: {1: 5568, 2: 5624, 3: 808}
Examples: ['2 BHK apartment -> New residential community -> Ready-to-move apartment', 'Home renovation', 'Property for investment -> Commercial plot', 'Retail space']


In [5]:
# ---- Feature-type assignment (verified against the output above) -------------------------
OHE_COLS     = ["location", "job_role", "lead_source"]          # low-cardinality nominal
ORDINAL_COL  = "email_response"                                  # genuinely ordered
HC_COL       = "preferred_location"                              # high-cardinality nominal
TEXT_COL     = "item_checked"                                    # sequence-like text, items joined by '->'
COUNT_COL    = "demo_requested"                                  # count 0-4 -> binary flag + numeric count
NUM_COLS     = ["age", "website_visits", "pages_viewed", "time_on_website", "days_since_last_interaction",
                "avg_response_time", "previous_purchases", "budget", "lead_score", "annual_cost_of_living"]

feature_groups = {
    "ID (drop)": [ID_COL], "Target": [TARGET],
    "Low-cardinality nominal": OHE_COLS, "Ordinal": [ORDINAL_COL],
    "High-cardinality nominal": [HC_COL], "Text / sequence": [TEXT_COL],
    "Binary-from-count": [COUNT_COL], "Numerical": NUM_COLS,
}
assigned = [c for g in feature_groups.values() for c in g]
assert sorted(assigned) == sorted(df.columns), "Some columns are unassigned / double-assigned!"
display(pd.DataFrame({"group": list(feature_groups), "columns": [", ".join(v) for v in feature_groups.values()],
                      "n": [len(v) for v in feature_groups.values()]}))

,group,columns,n
0,ID (drop),lead_id,1
1,Target,conversion_status,1
2,Low-cardinality nominal,"location, job_role, lead_source",3
3,Ordinal,email_response,1
4,High-cardinality nominal,preferred_location,1
5,Text / sequence,item_checked,1
6,Binary-from-count,demo_requested,1
7,Numerical,"age, website_visits, pages_viewed, time_on_website, days_since_las...",10


**What the inspection tells us**

| Column | Finding | Consequence |
|---|---|---|
| `lead_id` | 12 000 unique values = one per row | Pure identifier → **drop** |
| `location`, `job_role`, `lead_source` | 13 / 15 / 12 unlabelled-order categories | Nominal, low cardinality → **One-Hot** |
| `email_response` | `No Response`, `Slow`, `Moderate`, `Fast` | A real engagement ladder → **Ordinal** |
| `preferred_location` | 86 localities (some city-specific, one name appears in two cities) | High cardinality → **Target + Frequency** (One-Hot would add 86 sparse columns) |
| `item_checked` | Not categorical: chains such as `2 BHK apartment -> 3 BHK apartment -> Gated community` | Treating the 49 strings as classes would hide that they share items → **tokenise on `->` + TF-IDF** |
| `demo_requested` | **Not binary** — values 0–4 (a count of demo requests) | Derive a true **0/1 flag** (`>0`) and keep the count as numeric |
| `conversion_status` | 0/1 | Target |
| other numerics | counts, durations, money | Scaled (budget log-transformed first) |

## 5. Target Variable Identification
`conversion_status` (1 = lead converted) is the supervised target. It is binary, so we will use a **stratified** split to keep the class ratio identical in train and test.

In [6]:
print("Target column:", TARGET)
print("Class counts :", df[TARGET].value_counts().sort_index().to_dict())
print("Class share  :", (df[TARGET].value_counts(normalize=True).sort_index().round(4)).to_dict())
X = df.drop(columns=[TARGET])
y = df[TARGET]
print("X shape:", X.shape, "| y shape:", y.shape)

Target column: conversion_status
Class counts : {0: 8760, 1: 3240}
Class share  : {0: 0.73, 1: 0.27}
X shape: (12000, 18) | y shape: (12000,)


## 6. Train/Test Split  (**before any target-based encoding**)
> **Where leakage could happen:** if a target encoder is fitted on the full dataset (or if TF-IDF / scaler statistics are computed on it), test rows influence the features the model trains on, and the test score becomes over-optimistic. The split therefore comes **first**, and every `.fit()` below receives training data only.

In [7]:
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

print("Train:", X_train_full.shape, "| Test:", X_test_full.shape)
print("Train target mean:", round(y_train.mean(), 4), "| Test target mean:", round(y_test.mean(), 4))
assert set(X_train_full.index).isdisjoint(X_test_full.index)
assert set(X_train_full[ID_COL]).isdisjoint(X_test_full[ID_COL])
print("Train / test rows are disjoint.  OK")

Train: (9600, 18) | Test: (2400, 18)
Train target mean: 0.27 | Test target mean: 0.27
Train / test rows are disjoint.  OK


## 7. Encoding Strategy
First the identifier is removed. We keep the ids separately only so the saved CSV can be traced back to the original leads — they are **not** features.

In [8]:
ids_train, ids_test = X_train_full[ID_COL].copy(), X_test_full[ID_COL].copy()
X_train = X_train_full.drop(columns=[ID_COL])
X_test  = X_test_full.drop(columns=[ID_COL])
print("lead_id dropped from features. Remaining feature columns:", X_train.shape[1])

strategy = pd.DataFrame([
    ("location",           "nominal, low-card",   "One-Hot",                        "no order, few categories -> no false ordering"),
    ("job_role",           "nominal, low-card",   "One-Hot",                        "no order, few categories"),
    ("lead_source",        "nominal, low-card",   "One-Hot",                        "no order, few categories"),
    ("email_response",     "ordinal",             "Ordinal (explicit order)",       "No Response < Slow < Moderate < Fast"),
    ("preferred_location", "nominal, high-card",  "Target (OOF) + Frequency",       "86 categories: One-Hot too wide"),
    ("item_checked",       "text / sequence",     "Tokenise on '->' + TF-IDF",      "items are shared across strings"),
    ("demo_requested",     "count 0-4",           "0/1 flag + numeric count",       "any-demo vs none is the strong signal"),
    ("lead_id",            "identifier",          "Dropped",                        "unique per row, no predictive meaning"),
    ("numeric columns",    "numerical",           "StandardScaler (budget: log1p)", "comparable scales, tames skew"),
], columns=["column", "type", "technique", "why"])
display(strategy)

lead_id dropped from features. Remaining feature columns:

 17


,column,type,technique,why
0,location,"nominal, low-card",One-Hot,"no order, few categories -> no false ordering"
1,job_role,"nominal, low-card",One-Hot,"no order, few categories"
2,lead_source,"nominal, low-card",One-Hot,"no order, few categories"
3,email_response,ordinal,Ordinal (explicit order),No Response < Slow < Moderate < Fast
4,preferred_location,"nominal, high-card",Target (OOF) + Frequency,86 categories: One-Hot too wide
5,item_checked,text / sequence,Tokenise on '->' + TF-IDF,items are shared across strings
6,demo_requested,count 0-4,0/1 flag + numeric count,any-demo vs none is the strong signal
7,lead_id,identifier,Dropped,"unique per row, no predictive meaning"
8,numeric columns,numerical,StandardScaler (budget: log1p),"comparable scales, tames skew"


## 8. One-Hot Encoding — `location`, `job_role`, `lead_source`
These are nominal (no natural order) with only 12–15 categories each, so One-Hot is ideal: no artificial ordering is introduced and the dimensionality stays small. `handle_unknown="ignore"` means a category never seen in training becomes an all-zero row instead of crashing.

In [9]:
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=float)   # sklearn >= 1.2
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False, dtype=float)           # older sklearn

def clean_names(names):
    return [re.sub(r"[^0-9a-zA-Z]+", "_", n).strip("_") for n in names]

ohe = make_ohe().fit(X_train[OHE_COLS])                       # FIT ON TRAIN ONLY
ohe_names = clean_names(ohe.get_feature_names_out(OHE_COLS))

def ohe_block(Xd):
    return pd.DataFrame(ohe.transform(Xd[OHE_COLS]), columns=ohe_names, index=Xd.index)

ohe_train, ohe_test = ohe_block(X_train), ohe_block(X_test)
print("One-Hot columns:", ohe_train.shape[1], "=", " + ".join(str(len(c)) for c in ohe.categories_))
display(ohe_train.head())

# behaviour on an unseen category: all-zero block for that column
demo = ohe.transform(pd.DataFrame({"location": ["Atlantis"], "job_role": ["Doctor"], "lead_source": ["Website"]}))
print("Unseen 'Atlantis' -> location one-hot sum =", demo[0, :len(ohe.categories_[0])].sum())

One-Hot columns: 40 = 13 + 15 + 12


,location_Ahmedabad,location_Bengaluru,location_Chennai,location_Delhi,location_Gurgaon,location_Hyderabad,location_Kochi,location_Kolkata,location_Mumbai,location_Noida,location_Pune,location_Vijayawada,location_Visakhapatnam,job_role_Architect,job_role_Business_Owner,job_role_Civil_Engineer,job_role_Company_Director,job_role_Consultant,job_role_Contractor,job_role_Doctor,job_role_Entrepreneur,job_role_Government_Employee,job_role_Investor,job_role_Manager,job_role_Project_Manager,job_role_Real_Estate_Developer,job_role_Sales_Manager,job_role_Software_Engineer,lead_source_Broker,lead_source_Direct_Inquiry,lead_source_Email_Campaign,lead_source_Exhibition,lead_source_Existing_Customer,lead_source_Facebook_Ads,lead_source_Google_Ads,lead_source_Instagram,lead_source_LinkedIn,lead_source_Property_Portal,lead_source_Referral,lead_source_Website
10505,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
7192,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
11406,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
11954,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8134,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


Unseen 'Atlantis' -> location one-hot sum = 0.0


## 9. Ordinal Encoding — `email_response`
The four categories describe how engaged the lead is: **No Response < Slow < Moderate < Fast**. That is a logically valid order, so an explicit `categories=[...]` list is passed (never rely on alphabetical order, which would give Fast=0, Moderate=1, No Response=2, Slow=3 — nonsense).

*Caveat:* ordinal encoding assumes equal spacing between levels; that is a modelling assumption. We sanity-check that the order is also monotone against the target **on the training set only** (using the test set for such a check would itself be a leak).

In [10]:
EMAIL_ORDER = ["No Response", "Slow", "Moderate", "Fast"]
assert set(EMAIL_ORDER) == set(df[ORDINAL_COL].unique()), "Order list does not match the real categories!"

ordenc = OrdinalEncoder(categories=[EMAIL_ORDER], dtype=float).fit(X_train[[ORDINAL_COL]])   # TRAIN ONLY
ORD_NAME = "email_response_ord"

def ord_block(Xd):
    return pd.DataFrame(ordenc.transform(Xd[[ORDINAL_COL]]), columns=[ORD_NAME], index=Xd.index)

ord_train, ord_test = ord_block(X_train), ord_block(X_test)

check = (pd.concat([X_train[ORDINAL_COL], ord_train[ORD_NAME], y_train], axis=1)
           .groupby(ORDINAL_COL).agg(code=(ORD_NAME, "first"), n=(TARGET, "size"), train_conversion_rate=(TARGET, "mean"))
           .sort_values("code"))
display(check)
print("Conversion rate rises with the ordinal code (train only):",
      bool(check["train_conversion_rate"].is_monotonic_increasing))

,code,n,train_conversion_rate
email_response,,,
No Response,0.0,1317,0.127563
Slow,1.0,2332,0.237136
Moderate,2.0,2841,0.281591
Fast,3.0,3110,0.344373


Conversion rate rises with the ordinal code (train only): True


## 10. Binary (0/1) Encoding — `demo_requested`
The column is a **count (0–4)**, so it is not binary as supplied. We create:
* `demo_requested_flag` — a genuine 0/1 variable (1 = at least one demo requested). This is a deterministic row-wise rule, so it needs no fitting and cannot leak.
* `demo_requested_count` — the original count, kept as a numeric feature (scaled later).

In [11]:
def binary_block(Xd):
    return pd.DataFrame({"demo_requested_flag": (Xd[COUNT_COL] > 0).astype(int)}, index=Xd.index)

bin_train, bin_test = binary_block(X_train), binary_block(X_test)
display(pd.crosstab(X_train[COUNT_COL], bin_train["demo_requested_flag"], rownames=["demo_requested"], colnames=["flag"]))
print("Train conversion rate by flag:", y_train.groupby(bin_train["demo_requested_flag"]).mean().round(3).to_dict())

flag,0,1
demo_requested,,
0,7125,0
1,0,1390
2,0,662
3,0,297
4,0,126


Train conversion rate by flag: {0: 0.183, 1: 0.52}


## 11. Target Encoding — `preferred_location` (leakage-safe)
Each category is replaced by a **smoothed** mean of the target:

$$\text{enc}(c)=\frac{\Sigma_c + m\cdot\bar y}{n_c + m}$$

with `m = 20` pulling rare categories toward the global mean $\bar y$.

**Leakage-safe design**
1. Split already done → the mapping is learned from **training rows only**.
2. **Training rows are encoded out-of-fold (5-fold, stratified):** a row's value comes from a mapping that never saw that row's own label. (Encoding a training row with a mean that includes its own target is the classic leak — the model learns to read the label back out of the feature.)
3. **Test rows** are encoded with the mapping from the *whole* training set; unseen categories fall back to the training global mean. Test labels are never used.

In [12]:
TE_SMOOTH = 20

def fit_target_map(cat, target, m=TE_SMOOTH):
    prior = target.mean()
    stats = target.groupby(cat).agg(["sum", "count"])
    return (stats["sum"] + m * prior) / (stats["count"] + m), prior

def apply_target_map(cat, mapping, prior):
    return cat.map(mapping).astype(float).fillna(prior)

def oof_target_encode(cat, target, n_splits=5, m=TE_SMOOTH, seed=RANDOM_STATE):
    out = np.zeros(len(cat))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for fit_idx, val_idx in skf.split(cat, target):
        mp, pr = fit_target_map(cat.iloc[fit_idx], target.iloc[fit_idx], m)
        out[val_idx] = apply_target_map(cat.iloc[val_idx], mp, pr).values
    return pd.Series(out, index=cat.index)

TE_NAME = "preferred_location_target_enc"
te_train = oof_target_encode(X_train[HC_COL], y_train).rename(TE_NAME).to_frame()          # out-of-fold
te_map, te_prior = fit_target_map(X_train[HC_COL], y_train)                                  # train-only mapping
te_test = apply_target_map(X_test[HC_COL], te_map, te_prior).rename(TE_NAME).to_frame()    # test: mapping only

unseen = set(X_test[HC_COL]) - set(X_train[HC_COL])
print("Global prior (train):", round(te_prior, 4), "| unseen test categories:", len(unseen))
print("\nHighest / lowest smoothed encodings learned from TRAIN:")
display(pd.concat([te_map.sort_values().head(4), te_map.sort_values().tail(4)]).rename("encoding").to_frame())

# Why OOF matters: naive in-sample encoding looks 'better' on train only because it has seen the labels
naive_train = apply_target_map(X_train[HC_COL], te_map, te_prior)
print("\nTrain AUC of the feature  -  naive in-sample: %.4f   vs   out-of-fold: %.4f"
      % (roc_auc_score(y_train, naive_train), roc_auc_score(y_train, te_train[TE_NAME])))
print("Test  AUC of the feature  -  %.4f   (the honest number; close to OOF, not to naive)"
      % roc_auc_score(y_test, te_test[TE_NAME]))

Global prior (train): 0.27 | unseen test categories: 0

Highest / lowest smoothed encodings learned from TRAIN:


,encoding
preferred_location,
Salt Lake,0.189474
Navi Mumbai,0.195580
Navrangpura,0.200000
Sohna Road,0.201587
Bandra,0.350000
Sector 18,0.359494
Ballygunge,0.361290
Hitech City,0.397468



Train AUC of the feature  -  naive in-sample: 0.5587   vs   out-of-fold: 0.4993
Test  AUC of the feature  -  0.5130   (the honest number; close to OOF, not to naive)


## 12. Frequency Encoding — `preferred_location`
Each category is replaced by its **relative frequency in the training set**. It uses no target information, so it cannot leak the label; it only needs to be fitted on train (counts from test rows would also be a — milder — leak). Unseen categories get 0.

Below we compare the high-cardinality options on `preferred_location`.

In [13]:
freq_map = X_train[HC_COL].value_counts(normalize=True)                 # TRAIN counts only
FREQ_NAME = "preferred_location_freq_enc"
freq_train = X_train[HC_COL].map(freq_map).fillna(0.0).rename(FREQ_NAME).to_frame()
freq_test  = X_test[HC_COL].map(freq_map).fillna(0.0).rename(FREQ_NAME).to_frame()
display(freq_train.join(X_train[HC_COL]).drop_duplicates(HC_COL).sort_values(FREQ_NAME).iloc[[0, 1, -2, -1]])

# --- Compare approaches for the high-cardinality column --------------------------------
be = ce.BinaryEncoder(cols=[HC_COL]).fit(X_train[[HC_COL]])             # base-2 encoding (extra option)
be_cols = be.transform(X_train[[HC_COL]]).shape[1]

comparison = pd.DataFrame({
    "output columns": {"One-Hot": X_train[HC_COL].nunique(), "Binary-base (category_encoders)": be_cols,
                       "Target (OOF)": 1, "Frequency": 1},
    "uses target?": {"One-Hot": "no", "Binary-base (category_encoders)": "no", "Target (OOF)": "YES (train only, OOF)", "Frequency": "no"},
    "train corr with y": {"One-Hot": np.nan, "Binary-base (category_encoders)": np.nan,
                          "Target (OOF)": np.corrcoef(te_train[TE_NAME], y_train)[0, 1],
                          "Frequency": np.corrcoef(freq_train[FREQ_NAME], y_train)[0, 1]},
    "test AUC (single feature)": {"One-Hot": np.nan, "Binary-base (category_encoders)": np.nan,
                                  "Target (OOF)": roc_auc_score(y_test, te_test[TE_NAME]),
                                  "Frequency": roc_auc_score(y_test, freq_test[FREQ_NAME])},
})
display(comparison.round(4))
print("Binary-base encoding is shown for comparison only (arbitrary bit patterns create false similarity); "
      "Target + Frequency are kept in the final matrix.")

,preferred_location_freq_enc,preferred_location
7136,0.002812,Madhurawada
8340,0.002917,Park Street
1395,0.024583,Borivali
3404,0.028229,Goregaon


,output columns,uses target?,train corr with y,test AUC (single feature)
One-Hot,86,no,NaN,NaN
Binary-base (category_encoders),7,no,NaN,NaN
Target (OOF),1,"YES (train only, OOF)",0.0056,0.513
Frequency,1,no,-0.0030,0.460


Binary-base encoding is shown for comparison only (arbitrary bit patterns create false similarity); Target + Frequency are kept in the final matrix.


**Reading the comparison:** One-Hot would add 86 mostly-sparse columns; Binary-base shrinks that to 7 but its bit patterns are arbitrary. Target and Frequency each use **one** column. On this dataset both are near chance (correlation with the target ≈ 0, single-feature test AUC ≈ 0.5), i.e. `preferred_location` carries very little signal for conversion once encoded honestly. The *naive* in-sample target encoding looks better on train (AUC above) only because it has seen the labels — exactly the illusion out-of-fold encoding removes. Both encodings are kept because they capture different things (conversion propensity vs. how common the locality is) and cost one column each; drop them if you want a leaner model.

## 13. TF-IDF / Text Feature Extraction — `item_checked`
Each value is an ordered chain of items (`A -> B -> C`). Using the 49 whole strings as classes would make `"2 BHK apartment"` and `"2 BHK apartment -> 3 BHK apartment"` look unrelated. Instead we:
1. **Tokenise on `->`** so each *item* is one token (a plain whitespace tokenizer would wrongly split "2 BHK apartment" into three words).
2. Apply **TF-IDF** — rare items get higher weight, and each row is L2-normalised so long chains do not dominate. The vocabulary and IDF weights are learned from **training rows only**.
3. Add `n_items` (chain length) as an extra numeric feature.

In [14]:
def item_tokenizer(s):
    return [t.strip() for t in s.split("->") if t.strip()]

tfidf = TfidfVectorizer(tokenizer=item_tokenizer, token_pattern=None, lowercase=False, norm="l2")
tfidf.fit(X_train[TEXT_COL])                                              # vocabulary + IDF from TRAIN only
tfidf_names = ["item_" + n for n in clean_names(tfidf.get_feature_names_out())]

def tfidf_block(Xd):
    return pd.DataFrame(tfidf.transform(Xd[TEXT_COL]).toarray(), columns=tfidf_names, index=Xd.index)

tfidf_train, tfidf_test = tfidf_block(X_train), tfidf_block(X_test)
print("TF-IDF features:", tfidf_train.shape[1], "(= distinct items seen in train)")

idf = pd.Series(tfidf.idf_, index=tfidf_names).sort_values()
print("\nMost common items (lowest IDF) and rarest items (highest IDF):")
display(pd.concat([idf.head(3), idf.tail(3)]).rename("idf").to_frame())

i = X_train.index[3]
row = tfidf_train.loc[i]
print("\nExample row:", X_train.loc[i, TEXT_COL])
display(row[row > 0].round(4).to_frame("tf-idf weight"))

# alternatives, for dimensionality comparison only
multihot = CountVectorizer(tokenizer=item_tokenizer, token_pattern=None, lowercase=False, binary=True).fit(X_train[TEXT_COL])
print("\nColumns -> whole-string One-Hot: %d | multi-hot items: %d | TF-IDF items: %d (+1 n_items)"
      % (X_train[TEXT_COL].nunique(), len(multihot.get_feature_names_out()), tfidf_train.shape[1]))

TF-IDF features: 28 (= distinct items seen in train)

Most common items (lowest IDF) and rarest items (highest IDF):


,idf
item_2_BHK_apartment,2.876605
item_3_BHK_apartment,2.909100
item_Property_for_investment,2.994898
item_Civil_engineering_consultation,4.496299
item_Infrastructure_project,4.571201
item_Architectural_design,4.586126



Example row: 2 BHK apartment -> New residential community -> Ready-to-move apartment


,tf-idf weight
item_2_BHK_apartment,0.4717
item_New_residential_community,0.6573
item_Ready_to_move_apartment,0.5878



Columns -> whole-string One-Hot: 49 | multi-hot items: 28 | TF-IDF items: 28 (+1 n_items)


## 14. Combine Encoded Features
Numeric columns are prepared too: `budget` is extremely right-skewed (max ≈ 600 M vs median ≈ 12 M) so it gets `log1p`, then all numeric columns (plus `demo_requested_count` and `n_items`) are standardised with a `StandardScaler` **fitted on train only**.

In [15]:
def numeric_raw(Xd):
    n = Xd[NUM_COLS].astype(float).copy()
    n["budget"] = np.log1p(n["budget"])
    n["demo_requested_count"] = Xd[COUNT_COL].astype(float)
    n["n_items"] = Xd[TEXT_COL].map(lambda s: len(item_tokenizer(s))).astype(float)
    return n

scaler = StandardScaler().fit(numeric_raw(X_train))                      # TRAIN ONLY
def numeric_block(Xd):
    raw = numeric_raw(Xd)
    return pd.DataFrame(scaler.transform(raw), columns=raw.columns, index=raw.index)

num_train, num_test = numeric_block(X_train), numeric_block(X_test)

X_train_final = pd.concat([num_train, bin_train, ohe_train, ord_train, te_train, freq_train, tfidf_train], axis=1)
X_test_final  = pd.concat([num_test,  bin_test,  ohe_test,  ord_test,  te_test,  freq_test,  tfidf_test],  axis=1)

groups = {"numeric (scaled)": num_train.shape[1], "binary flag": bin_train.shape[1], "one-hot": ohe_train.shape[1],
          "ordinal": ord_train.shape[1], "target enc": te_train.shape[1], "frequency enc": freq_train.shape[1],
          "tf-idf items": tfidf_train.shape[1]}
display(pd.Series(groups, name="n_columns").to_frame().T)
print("Total feature columns:", sum(groups.values()))
assert list(X_train_final.columns) == list(X_test_final.columns)
assert X_train_final.index.equals(X_train.index) and X_test_final.index.equals(X_test.index)

,numeric (scaled),binary flag,one-hot,ordinal,target enc,frequency enc,tf-idf items
n_columns,12,1,40,1,1,1,28


Total feature columns: 84


## 15. Final Feature Matrix

In [16]:
print("X_train_final:", X_train_final.shape, "| y_train:", y_train.shape)
print("X_test_final :", X_test_final.shape,  "| y_test :", y_test.shape)
assert len(X_train_final) == len(y_train) and len(X_test_final) == len(y_test)
print("\nDtypes:", X_train_final.dtypes.value_counts().to_dict())
display(X_train_final.head())

X_train_final: (9600, 84) | y_train: (9600,)
X_test_final : (2400, 84) | y_test : (2400,)

Dtypes: {dtype('float64'): 83, dtype('int64'): 1}


,age,website_visits,pages_viewed,time_on_website,days_since_last_interaction,avg_response_time,previous_purchases,budget,lead_score,annual_cost_of_living,demo_requested_count,n_items,demo_requested_flag,location_Ahmedabad,location_Bengaluru,location_Chennai,location_Delhi,location_Gurgaon,location_Hyderabad,location_Kochi,location_Kolkata,location_Mumbai,location_Noida,location_Pune,location_Vijayawada,location_Visakhapatnam,job_role_Architect,job_role_Business_Owner,job_role_Civil_Engineer,job_role_Company_Director,job_role_Consultant,job_role_Contractor,job_role_Doctor,job_role_Entrepreneur,job_role_Government_Employee,job_role_Investor,job_role_Manager,job_role_Project_Manager,job_role_Real_Estate_Developer,job_role_Sales_Manager,job_role_Software_Engineer,lead_source_Broker,lead_source_Direct_Inquiry,lead_source_Email_Campaign,lead_source_Exhibition,lead_source_Existing_Customer,lead_source_Facebook_Ads,lead_source_Google_Ads,lead_source_Instagram,lead_source_LinkedIn,lead_source_Property_Portal,lead_source_Referral,lead_source_Website,email_response_ord,preferred_location_target_enc,preferred_location_freq_enc,item_1_BHK_apartment,item_2_BHK_apartment,item_3_BHK_apartment,item_4_BHK_apartment,item_Architectural_design,item_Building_construction,item_Civil_engineering_consultation,item_Commercial_plot,item_Construction_project,item_Construction_project_management,item_Gated_community,item_Home_renovation,item_Independent_house,item_Industrial_property,item_Infrastructure_project,item_Interior_design,item_Land_for_construction,item_Luxury_apartment,item_New_residential_community,item_Office_space,item_Property_for_investment,item_Ready_to_move_apartment,item_Residential_plot,item_Retail_space,item_Structural_design,item_Under_construction_apartment,item_Villa,item_Warehouse
10505,0.879067,0.011927,0.538811,0.536718,-0.856887,-0.478896,-0.508936,-0.361567,0.394588,0.523714,-0.502387,0.652768,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,0.206792,0.016771,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.70804,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.706172,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
7192,0.434257,-0.318041,-0.531285,-0.761861,3.287335,-0.613948,-0.508936,-0.456901,-2.046098,-0.259565,-0.502387,-0.988556,0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.234349,0.022187,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.00000,0.0,0.0,0.0,0.000000,0.0,1.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
11406,-0.344160,-1.802896,-1.499467,-1.206182,-0.254529,-0.648373,-0.508936,-1.353624,-1.156502,-0.971637,-0.502387,0.652768,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.203296,0.005833,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.639571,0.00000,0.0,0.0,0.0,0.768732,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
11954,-0.900173,0.671863,0.385940,0.140003,-0.278623,0.217549,-0.508936,-1.176634,-0.000788,0.274489,-0.502387,2.294092,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.246966,0.012917,0.000000,0.471681,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.657265,0.0,0.0,0.587809,0.0,0.0,0.0,0.0,0.0,0.0
8134,-0.900173,-0.812993,-1.040854,-0.708966,-0.905075,-0.621892,-0.508936,-0.732405,-0.867573,-1.327673,-0.502387,0.652768,0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.334635,0.004792,0.810709,0.585450,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.00000,0.0,0.0,0.0,0.

## 16. Check for Missing Values

In [17]:
for name, frame in [("X_train_final", X_train_final), ("X_test_final", X_test_final)]:
    n_nan = int(frame.isna().sum().sum())
    n_inf = int(np.isinf(frame.values).sum())
    n_obj = int((frame.dtypes == object).sum())
    print(f"{name}: NaN={n_nan}  Inf={n_inf}  non-numeric columns={n_obj}")
    assert n_nan == 0 and n_inf == 0 and n_obj == 0
assert y_train.isna().sum() == 0 and y_test.isna().sum() == 0
print("\nNo missing / infinite / non-numeric values anywhere.  OK")

X_train_final: NaN=0  Inf=0  non-numeric columns=0
X_test_final: NaN=0  Inf=0  non-numeric columns=0

No missing / infinite / non-numeric values anywhere.  OK


## 17. Check for Data Leakage
Instead of just asserting the design is safe, we **test** it:

1. Structural checks (disjoint rows, no target or id in features).
2. Every fitted object learned its statistics from train, not from the full data.
3. **Reproducibility check:** a single function `build_features(train, y_train, test)` re-creates the whole matrix from scratch and must match the matrix above exactly.
4. **Test-influence check:** if we feed that function only 50 of the test rows (or different test rows), the *training* matrix must be identical — proving test data cannot influence anything fitted — and each test row must encode identically regardless of the other test rows.
5. A demonstration of what target-encoding leakage looks like, using `lead_id` as the extreme example.

In [18]:
# 1) structural checks ----------------------------------------------------------------------
assert set(X_train_final.index).isdisjoint(X_test_final.index)
assert TARGET not in X_train_final.columns and not any("lead" in c and "id" in c for c in X_train_final.columns)
assert not any(c.startswith("conversion") for c in X_train_final.columns)
print("1) Disjoint rows; target and lead_id are not among the features.  OK")

# 2) fitted objects were learned from train only --------------------------------------------
full_pref = set(df[HC_COL])
print("2) one-hot categories == train categories:",
      all(list(c) == sorted(X_train[col].unique()) for c, col in zip(ohe.categories_, OHE_COLS)))
print("   TF-IDF vocabulary size (train) =", len(tfidf.vocabulary_), "| scaler mean(age) train vs full-data:",
      round(scaler.mean_[0], 4), "vs", round(df['age'].mean(), 4), "(different -> test rows were NOT used)")
print("   target-encoding mapping built from", int(X_train[HC_COL].notna().sum()), "train rows; test labels never passed to any fit.")

# 3) from-scratch pipeline function ---------------------------------------------------------
def build_features(Xtr_full, ytr, Xte_full):
    Xtr = Xtr_full.drop(columns=[ID_COL], errors="ignore"); Xte = Xte_full.drop(columns=[ID_COL], errors="ignore")
    o = make_ohe().fit(Xtr[OHE_COLS]); on = clean_names(o.get_feature_names_out(OHE_COLS))
    od = OrdinalEncoder(categories=[EMAIL_ORDER], dtype=float).fit(Xtr[[ORDINAL_COL]])
    mp, pr = fit_target_map(Xtr[HC_COL], ytr)
    fm = Xtr[HC_COL].value_counts(normalize=True)
    tf = TfidfVectorizer(tokenizer=item_tokenizer, token_pattern=None, lowercase=False, norm="l2").fit(Xtr[TEXT_COL])
    tn = ["item_" + n for n in clean_names(tf.get_feature_names_out())]
    sc = StandardScaler().fit(numeric_raw(Xtr))

    def block(Xd, te_series):
        raw = numeric_raw(Xd)
        parts = [pd.DataFrame(sc.transform(raw), columns=raw.columns, index=Xd.index),
                 binary_block(Xd),
                 pd.DataFrame(o.transform(Xd[OHE_COLS]), columns=on, index=Xd.index),
                 pd.DataFrame(od.transform(Xd[[ORDINAL_COL]]), columns=[ORD_NAME], index=Xd.index),
                 te_series.rename(TE_NAME).to_frame(),
                 Xd[HC_COL].map(fm).fillna(0.0).rename(FREQ_NAME).to_frame(),
                 pd.DataFrame(tf.transform(Xd[TEXT_COL]).toarray(), columns=tn, index=Xd.index)]
        return pd.concat(parts, axis=1)

    return (block(Xtr, oof_target_encode(Xtr[HC_COL], ytr)),
            block(Xte, apply_target_map(Xte[HC_COL], mp, pr)))

F_tr, F_te = build_features(X_train_full, y_train, X_test_full)
assert np.allclose(F_tr.values, X_train_final.values) and np.allclose(F_te.values, X_test_final.values)
print("3) build_features() reproduces the step-by-step matrices exactly.  OK")

# 4) test-influence check -------------------------------------------------------------------
F_tr_small, F_te_small = build_features(X_train_full, y_train, X_test_full.iloc[:50])
assert np.allclose(F_tr_small.values, X_train_final.values), "Test data changed the TRAIN features!"
assert np.allclose(F_te_small.values, X_test_final.iloc[:50].values), "A test row's encoding depends on other test rows!"
print("4) Train matrix is identical whether all %d or only 50 test rows are supplied; "
      "test rows encode independently of each other.  OK" % len(X_test_full))

# 5) what leakage looks like: target-encode the unique lead_id --------------------------------
mp_id, pr_id = fit_target_map(X_train_full[ID_COL], y_train)
naive_id = apply_target_map(X_train_full[ID_COL], mp_id, pr_id)
oof_id = oof_target_encode(X_train_full[ID_COL], y_train)
print("\n5) lead_id demo  (each id is unique, so a naive mean contains the row's own label)")
print("   naive in-sample encoding : corr with y = %.3f   <- the label is leaked into the feature" % np.corrcoef(naive_id, y_train)[0, 1])
print("   out-of-fold encoding     : only %d distinct values (each id is unseen in its fold -> just the fold's global mean), "
      "corr with y = %.3f -> no information, no leak" % (oof_id.nunique(), np.corrcoef(oof_id, y_train)[0, 1]))

1) Disjoint rows; target and lead_id are not among the features.  OK
2) one-hot categories == train categories: True
   TF-IDF vocabulary size (train) = 28 | scaler mean(age) train vs full-data: 38.0949 vs 38.0539 (different -> test rows were NOT used)
   target-encoding mapping built from 9600 train rows; test labels never passed to any fit.


3) build_features() reproduces the step-by-step matrices exactly.  OK


4) Train matrix is identical whether all 2400 or only 50 test rows are supplied; test rows encode independently of each other.  OK

5) lead_id demo  (each id is unique, so a naive mean contains the row's own label)
   naive in-sample encoding : corr with y = 1.000   <- the label is leaked into the feature
   out-of-fold encoding     : only 2 distinct values (each id is unseen in its fold -> just the fold's global mean), corr with y = -0.001 -> no information, no leak


**Residual caveat (outside the encoding itself):** `lead_score` is a pre-computed score with the highest correlation to the target. If, in your business process, it is computed *after* (or using) the conversion outcome, it would be a leakage feature and should be removed; if it is computed from earlier behaviour it is legitimate. It is kept here because nothing in the data says otherwise — please confirm with the data owner.

## 18. Display Final Encoded Dataset

In [19]:
train_out = pd.concat([ids_train.rename("lead_id"), X_train_final, y_train], axis=1).assign(split="train")
test_out  = pd.concat([ids_test.rename("lead_id"),  X_test_final,  y_test],  axis=1).assign(split="test")
processed = pd.concat([train_out, test_out]).sort_index()
processed = processed[["lead_id", "split"] + list(X_train_final.columns) + [TARGET]]

print("Final processed dataset:", processed.shape, "  (lead_id kept only for traceability - NOT a feature)")
display(processed.head(10))
print("\nTrain feature summary (first 12 columns):")
display(X_train_final.iloc[:, :12].describe().T.round(3))

Final processed dataset: (12000, 87)   (lead_id kept only for traceability - NOT a feature)


,lead_id,split,age,website_visits,pages_viewed,time_on_website,days_since_last_interaction,avg_response_time,previous_purchases,budget,lead_score,annual_cost_of_living,demo_requested_count,n_items,demo_requested_flag,location_Ahmedabad,location_Bengaluru,location_Chennai,location_Delhi,location_Gurgaon,location_Hyderabad,location_Kochi,location_Kolkata,location_Mumbai,location_Noida,location_Pune,location_Vijayawada,location_Visakhapatnam,job_role_Architect,job_role_Business_Owner,job_role_Civil_Engineer,job_role_Company_Director,job_role_Consultant,job_role_Contractor,job_role_Doctor,job_role_Entrepreneur,job_role_Government_Employee,job_role_Investor,job_role_Manager,job_role_Project_Manager,job_role_Real_Estate_Developer,job_role_Sales_Manager,job_role_Software_Engineer,lead_source_Broker,lead_source_Direct_Inquiry,lead_source_Email_Campaign,lead_source_Exhibition,lead_source_Existing_Customer,lead_source_Facebook_Ads,lead_source_Google_Ads,lead_source_Instagram,lead_source_LinkedIn,lead_source_Property_Portal,lead_source_Referral,lead_source_Website,email_response_ord,preferred_location_target_enc,preferred_location_freq_enc,item_1_BHK_apartment,item_2_BHK_apartment,item_3_BHK_apartment,item_4_BHK_apartment,item_Architectural_design,item_Building_construction,item_Civil_engineering_consultation,item_Commercial_plot,item_Construction_project,item_Construction_project_management,item_Gated_community,item_Home_renovation,item_Independent_house,item_Industrial_property,item_Infrastructure_project,item_Interior_design,item_Land_for_construction,item_Luxury_apartment,item_New_residential_community,item_Office_space,item_Property_for_investment,item_Ready_to_move_apartment,item_Residential_plot,item_Retail_space,item_Structural_design,item_Under_construction_apartment,item_Villa,item_Warehouse,conversion_status
0,LD009064,train,0.545460,1.661766,0.946467,1.004841,-0.543661,3.024514,-0.508936,-0.197133,0.250124,-0.259565,-0.502387,0.652768,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.290655,0.015938,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.639571,0.000000,0.0,0.0,0.000000,0.768732,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,1
1,LD007452,train,-0.121755,-0.483025,-0.582242,-0.804178,-0.808698,1.059639,-0.508936,-0.247753,-1.270553,0.452507,-0.502387,-0.988556,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.244837,0.011771,0.000000,1.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0
2,LD008371,train,0.879067,-0.483025,-0.225543,-0.198527,0.853810,-0.423286,-0.508936,0.143444,-0.160459,0.523714,-0.502387,-0.988556,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,0.314685,0.024583,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,1.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0
3,LD001858,train,-0.232958,1.661766,2.271347,3.744816,-0.953264,0.048072,-0.508936,-0.299875,1.504681,0.559318,-0.502387,0.652768,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.303141,0.006771,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.642583,0.0,0.766216,0.0,0.000000,0.0,0.000000,0.000000,1
4,LD000248,test,0.767865,-0.153057,0.233070,0.073884,0.251452,2.065909,-0.508936,-1.387200,0.675913,1.591822,0.671357,2.294092,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Train feature summary (first 12 columns):


,count,mean,std,min,25%,50%,75%,max
age,9600.0,0.0,1.0,-1.790,-0.678,-0.011,0.657,3.326
website_visits,9600.0,-0.0,1.0,-1.803,-0.813,-0.153,0.672,3.972
pages_viewed,9600.0,-0.0,1.0,-1.601,-0.735,-0.175,0.590,5.074
time_on_website,9600.0,-0.0,1.0,-1.405,-0.743,-0.218,0.508,6.022
days_since_last_interaction,9600.0,0.0,1.0,-1.026,-0.736,-0.327,0.420,3.311
avg_response_time,9600.0,0.0,1.0,-0.656,-0.598,-0.442,0.186,3.787
previous_purchases,9600.0,0.0,1.0,-0.509,-0.509,-0.509,1.024,5.621
budget,9600.0,0.0,1.0,-3.591,-0.576,-0.006,0.661,3.528
lead_score,9600.0,-0.0,1.0,-2.548,-0.716,-0.084,0.661,4.067
annual_cost_of_living,9600.0,-0.0,1.0,-1.862,-0.651,-0.153,0.453,4.939


## 19. Save the Processed Dataset

In [20]:
OUT_DIR = os.path.join(os.getcwd(), "processed_output")
os.makedirs(OUT_DIR, exist_ok=True)

paths = {"processed_dataset.csv": processed,
         "processed_train.csv": processed[processed.split == "train"],
         "processed_test.csv":  processed[processed.split == "test"]}
for fname, frame in paths.items():
    frame.to_csv(os.path.join(OUT_DIR, fname), index=False)

for fname, frame in paths.items():                                   # verify by reading back
    back = pd.read_csv(os.path.join(OUT_DIR, fname))
    assert back.shape == frame.shape, fname
    print(f"saved {fname:<24} shape={back.shape}  size={os.path.getsize(os.path.join(OUT_DIR, fname))/1e6:.2f} MB")

try:                                                                  # auto-download when running in Colab
    from google.colab import files
    files.download(os.path.join(OUT_DIR, "processed_dataset.csv"))
except ImportError:
    print("(Not in Colab: files are in", OUT_DIR + ")")

saved processed_dataset.csv    shape=(12000, 87)  size=7.01 MB
saved processed_train.csv      shape=(9600, 87)  size=5.61 MB
saved processed_test.csv       shape=(2400, 87)  size=1.40 MB
(Not in Colab: files are in /home/claude/nb/processed_output)


## 20. Summary of Encoding Techniques Used
All cardinalities below are computed from your dataset.

In [21]:
nunq = df.nunique()
rows = [
 ("location",           "Categorical (nominal)",  nunq["location"],           "One-Hot",                          f"Low-cardinality nominal feature -> {len(ohe.categories_[0])} columns"),
 ("job_role",           "Categorical (nominal)",  nunq["job_role"],           "One-Hot",                          f"Low-cardinality nominal feature -> {len(ohe.categories_[1])} columns"),
 ("lead_source",        "Categorical (nominal)",  nunq["lead_source"],        "One-Hot",                          f"Nominal feature -> {len(ohe.categories_[2])} columns"),
 ("email_response",     "Ordinal",                nunq["email_response"],     "Ordinal (No Response<Slow<Moderate<Fast)", "Natural engagement ordering; order verified on train"),
 ("preferred_location", "High-cardinality",       nunq["preferred_location"], "Target (OOF, smoothed) + Frequency", "Many categories; One-Hot would add %d sparse columns" % nunq["preferred_location"]),
 ("item_checked",       "Text / sequence",        nunq["item_checked"],       "Split on '->' + TF-IDF (+ n_items)", f"{len(all_items)} distinct items inside {nunq['item_checked']} strings -> {tfidf_train.shape[1]} TF-IDF cols"),
 ("demo_requested",     "Count 0-4 (not truly binary)", nunq["demo_requested"], "0/1 flag + scaled count",        "Flag = any demo requested; count kept numerically"),
 ("lead_id",            "ID",                     nunq["lead_id"],            "Dropped",                          "Unique identifier, no predictive meaning (and would leak under target encoding)"),
 ("conversion_status",  "Binary target",          nunq["conversion_status"],  "Not encoded (target y)",           "Kept as y; used only to fit train-side target encoding"),
]
for c in NUM_COLS:
    why = "log1p (right-skewed) then StandardScaler" if c == "budget" else "StandardScaler (fit on train)"
    rows.append((c, "Numerical", nunq[c], "Scaling", why))

summary = pd.DataFrame(rows, columns=["Column", "Data Type", "Cardinality", "Encoding Technique", "Reason"])
display(summary)
assert set(summary["Column"]) == set(df.columns)
print("Every one of the", df.shape[1], "original columns is accounted for.")
print("Final matrix: %d train rows x %d features, %d test rows x %d features."
      % (*X_train_final.shape, *X_test_final.shape))

,Column,Data Type,Cardinality,Encoding Technique,Reason
0,location,Categorical (nominal),13,One-Hot,Low-cardinality nominal feature -> 13 columns
1,job_role,Categorical (nominal),15,One-Hot,Low-cardinality nominal feature -> 15 columns
2,lead_source,Categorical (nominal),12,One-Hot,Nominal feature -> 12 columns
3,email_response,Ordinal,4,Ordinal (No Response<Slow<Moderate<Fast),Natural engagement ordering; order verified on train
4,preferred_location,High-cardinality,86,"Target (OOF, smoothed) + Frequency",Many categories; One-Hot would add 86 sparse columns
5,item_checked,Text / sequence,49,Split on '->' + TF-IDF (+ n_items),28 distinct items inside 49 strings -> 28 TF-IDF cols
6,demo_requested,Count 0-4 (not truly binary),5,0/1 flag + scaled count,Flag = any demo requested; count kept numerically
7,lead_id,ID,12000,Dropped,"Unique identifier, no predictive meaning (and would leak under tar..."
8,conversion_status,Binary target,2,Not encoded (target y),Kept as y; used only to fit train-side target encoding
9,age,Numerical,47,Scaling,StandardScaler (fit on train)


Every one of the 19 original columns is accounted for.
Final matrix: 9600 train rows x 84 features, 2400 test rows x 84 features.


### Takeaways
* **One-Hot** for `location`, `job_role`, `lead_source` — nominal, low cardinality.
* **Ordinal** for `email_response` with an explicit, logically valid order.
* **0/1 flag** for `demo_requested` — after discovering it is actually a 0–4 count.
* **Target (out-of-fold, smoothed) + Frequency** for the 86-category `preferred_location`.
* **Item tokenisation + TF-IDF** for `item_checked` instead of 49 meaningless classes.
* **`lead_id` dropped.**
* **No leakage:** split first; all `fit` calls use train only; training rows are target-encoded out-of-fold; section 17 verifies this empirically.